# Mersenne

Collect html pages for MERSENNE articles and their translations from https://comptes-rendus.academie-sciences.fr/, convert the html files to plain texts using `pandoc`, extract text blocks (section titles, abstracts, paragraphs and such, we refer to all these text blocks as paragraphs for simplicity.)

In [1]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import unicodedata

In [2]:
# define the global variable for data path
# the work folder
DATA_DIR = '../reproduce_MERSENNE'
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

META_PATH = '../resource_october_2024.tsv'

## 1. Request htmls


In [3]:
meta_df = pd.read_csv(META_PATH, sep = '\t', index_col = 0 )
meta_df.head()

,docid,Titre,Auteur,Série,Année,Vol.,Thém.,LO,Traducteur.trice,Version originale,Traduction
0,mersenne0,La science n’est pas floue,"Etienne Ghys , Ghislain de Marsily",CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
1,mersenne1,Réchauffement climatique : état des connaissan...,Valérie Masson-Delmotte,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
2,mersenne2,"Le changement climatique, une chance pour l’hu...",Mireille Delmas-Marty,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
3,mersenne3,Changement climatique et éducation,"Pierre Léna , David Wilgenbus",CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
4,mersenne4,« Science des données » versus science physiqu...,Venkatramani Balaji,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...


In [4]:

for i in range(len(meta_df)):
    store_dir = f"{DATA_DIR}/html/{meta_df.loc[i]['LO']}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    
    url = meta_df.loc[i]['Version originale']
    if url is None or url is np.nan :
        continue
    name = url.split('/')[-2:-1][0]
    store_path = f"{store_dir}/mersenne{i}.html"

    try:
        res = requests.get(url)
        if res.status_code == 200:
            with open(store_path , 'wb') as f:
                f.write(res.content)
    except:
        print(res.status_code)
        print(f'Failed to fetch {url}')



In [5]:

for i in range(len(meta_df)):
    # Translation
    store_dir = f'{DATA_DIR}/html/EN' if meta_df.loc[i]['LO']=='FR' else f'{DATA_DIR}/html/FR'
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    
    url = meta_df.loc[i]['Traduction']
    if url is None or str(url)=='nan':
        continue
    name = url.split('/')[-2:-1][0]
    store_path = f"{store_dir}/mersenne{i}.html"

    try:
        res = requests.get(url)
        if res.status_code == 200:
            with open(store_path , 'wb') as f:
                f.write(res.content)
    except:
        print(f'Failed to fetch {url}')
            

## 2. html2txt

In [6]:
TXT_DIR= f"{DATA_DIR}/html2txt"

for lang in ['EN', 'FR']:
    print(lang)
    html_dir = f'{DATA_DIR}/html/{lang}'
    store_dir = f'{TXT_DIR}/{lang}'
    Path(store_dir).mkdir(parents=True, exist_ok=True)

    for fname in os.listdir(f'{DATA_DIR}/html/EN'):
        fpath = f"{html_dir}/{fname}"
        if not os.path.exists(fpath):
            print(f'no file at {fpath}')
        
        os.system(f"pandoc -f html -t plain {fpath} --wrap=preserve -o {store_dir}/{fname}2txt.txt")


EN


[WARNING] Could not convert TeX math '\eta_{r} = \frac{\eta_{\text{app}}}{\eta_{l}},', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 + \frac{1.25{\mathit{Φ}}}{1 - \frac{\mathit{Φ}}{{\mathit{Φ}}_{m}}}} \right)^{- 2}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = e^{\frac{B{\mathit{Φ}}}{1 - \alpha{\mathit{Φ}}}}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 - \frac{\mathit{Φ}}{{\mathit{Φ}}_{m}}} \right)^{- 2}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 - \frac{\phi}{\phi_{m}}} \right)^{- B\phi_{m}}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 - \frac{\phi}{\phi_{m}}} \right)^{- B\phi_{m}}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \exp\left\{ {\left\lbrack \left( {2.5 + \frac{\phi}{\phi_{m} - \phi}} \right)^{\alpha} \right\rbrack \cdot \frac{\phi}{\phi_{m}}} \right\}', rendering as TeX
[WARNING] Could not convert TeX

FR


[WARNING] Could not convert TeX math '\eta_{r} = \frac{\eta_{\text{app}}}{\eta_{l}},', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 + \frac{1.25{\mathit{Φ}}}{1 - \frac{\mathit{Φ}}{{\mathit{Φ}}_{m}}}} \right)^{- 2}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = e^{\frac{B{\mathit{Φ}}}{1 - \alpha{\mathit{Φ}}}}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 - \frac{\mathit{Φ}}{{\mathit{Φ}}_{m}}} \right)^{- 2}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 - \frac{\phi}{\phi_{m}}} \right)^{- B\phi_{m}}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \left( {1 - \frac{\phi}{\phi_{m}}} \right)^{- B\phi_{m}}', rendering as TeX
[WARNING] Could not convert TeX math '\eta_{r} = \exp\left\{ {\left\lbrack \left( {2.5 + \frac{\phi}{\phi_{m} - \phi}} \right)^{\alpha} \right\rbrack \cdot \frac{\phi}{\phi_{m}}} \right\}', rendering as TeX
[WARNING] Could not convert TeX

## 3. pre-process

- NFC normalization
- remove '\xa0'


In [7]:


begin_info_list = [ 
   "Version originale du texte intégral\n(Proposez une traduction )",
    "Traduction du texte intégral\n(Proposez une traduction dans une autre langue)\n\nTexte intégral traduit par :\n\n",
    "Traduction du texte intégral\n(Proposez une traduction dans une autre langue)",
]


end_info_list = [
    "\n\nBibliographie\n\n",
    "Cité par\n\nCité par Sources :\n",
]

def process_txt(fpath, store_fpath, begin_info_list = [], end_info_list = []):
    txt = open(fpath,'r').read()
    txt = re.sub('\xa0', '', txt)
    txt = re.sub('\n\n+', '\n\n', txt)
    txt = unicodedata.normalize("NFC", txt)

    begin = False
    print(fpath)
    for info in begin_info_list:
        res = txt.split(info)
        if len(res) == 2:
            print('begin')
            txt = res[1].strip()
            if info == "Traduction du texte intégral\n(Proposez une traduction dans une autre langue)\n\nTexte intégral traduit par :\n\n":
                txt = txt.split('\n\n',2)[-1].strip()
            begin = True
            break

    end = False
    for info in end_info_list:
        res = txt.split(info)
        if len(res) == 2:
            print('end')
            txt = res[0].strip()
            end = True
            break
    if len(end_info_list) > 0 and len(begin_info_list) > 0 and (not begin or not end):
        print(f'nothing removed at the beginning or the end of the raw text in {fpath} ')

    with open(store_fpath , 'w') as f:
        f.write(txt)

In [8]:

TXT_DIR= f"{DATA_DIR}/html2txt"
STORE_DIR= f"{DATA_DIR}/processed_txt"

for lang in ['EN', 'FR']:
    print(lang)
    txt_dir = f'{TXT_DIR}/{lang}'
    store_dir = f'{STORE_DIR}/{lang}'
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    
    for fname in os.listdir(f'{DATA_DIR}/html/EN'):
        fpath = f"{txt_dir}/{fname}2txt.txt"
        if not os.path.exists(fpath):
            print(f'no file at {fpath}')
            continue

        store_fpath = f"{store_dir}/{fname}2txt.processed.txt"
        process_txt(fpath, store_fpath, begin_info_list, end_info_list)

EN
../reproduce_MERSENNE/html2txt/EN/mersenne2.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne10.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne11.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne3.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne4.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne16.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne8.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne20.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne21.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne9.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne17.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne5.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne14.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne6.html2txt.txt
begin
end
../reproduce_MERSENNE/html2txt/EN/mersenne22.html2tx

## 4. folders for segmentation

- one block per file (block = sequences separated by `\n\n` in the processed text, or isolated equations, or tables )
  the '\n' in each block is removed, if there is no structured representation inside the block.

- `[]` are removed (which are place holders for figures)

note that in mesrenne24, there'a a table separated by several \n\n thus stored in several files

In [9]:
# to_remove_list = [
#     '------------------------------------------------------------------------',
#     '[]',
# ]

def preprocess_to_segment(fpath, store_dir, lang ):
    r"""
        store blocks separated by \n\n from the given article in fpath into different folders, 
        and marking their order in the corresponding folder name
        place holder [] for figures are removed

        all tables follows a block of r"(Table\d+.|Tableau\d+.)"
        if a table contains \n\n, we assume that the caption is above the table
        this function may store potential list items located below a table together with the table.        
    """
    lang = lang.lower()
    table_pattern = re.compile(r"(Table\d+.|Tableau\d+.)")

    # remove place holder [] etc.
    # segments = [ l.strip() for l in open(fpath,'r').read().strip().split('\n\n') if l.strip() not in ['[]']]
    segments = [ l for l in open(fpath,'r').read().strip().split('\n\n') if l.strip() not in ['[]', '']]
    if segments[-1] == '------------------------------------------------------------------------':
        segments = segments[:-1]

    def keep_structure(txt):
        # '----' in segments[i] == True can be equations
        # bool(re.findall( '  +', segments[i] )) can be tables, list or other structured representations
        keep = '----' in txt or bool(re.findall( '  +', txt))
        return keep
        
    def store_txt(to_write, store_fpath):
        with open(store_fpath, 'w', encoding = 'utf-8') as f:
            f.write(to_write)
        
    # write
    meta_info = {'table': []}
    i = 0
    while i < len(segments):
        to_write = segments[i]

        # table
        is_table = bool(re.match( table_pattern, to_write )) and not ' ' in to_write
        if is_table:
            end = i+2
            while end +1 < len(segments) and keep_structure(segments[end +1]):
                # this can include potential list items located below the table, 
                end +=1
            meta_info['table'].append(f"{i}-{end}")

            # tableX
            store_txt(to_write, store_fpath= f"{store_dir}/to_segment{i}.{lang}")
            # caption
            store_txt(segments[i+1], store_fpath= f"{store_dir}/to_segment{i+1}.{lang}")
            # table
            suffix = '' if i+2 == end else f'-{end}'
            store_txt( '\n\n'.join(segments[i+2:end+1]), store_fpath= f"{store_dir}/to_segment{i+2}{suffix}.{lang}")
            i = end+1
        else:
            # need to concatenate
            if '\n' in to_write and not keep_structure(to_write):
                tmp = [l.strip() for l in to_write.split('\n') ]
                if len(tmp) == 2 and (tmp[1][:4] =='http' or tmp[0] in ['¹', '²','³','⁴', '⁵','⁶', '⁷' ]):
                    to_write = ''.join(tmp)
                else:
                    to_write = ' '.join(tmp)
    
            store_fpath = f"{store_dir}/to_segment{i}.{lang}"
            store_txt(to_write, store_fpath)
            i+=1
        
    return meta_info


In [10]:

meta_df = pd.read_csv(META_PATH, sep = '\t', index_col = 0 )

PROCESS_DIR = f"{DATA_DIR}/processed_txt"
FOLDER_DIR = f"{DATA_DIR}/to_segment_txt"
idx_pattern = re.compile(r"mersenne(\d+).+")



txt_dir = f'{PROCESS_DIR}/EN'  
for fname in os.listdir(txt_dir):
    idx = int(re.findall(idx_pattern, fname )[0])

    src_url = meta_df.loc[idx]['Version originale']
    tgt_url = meta_df.loc[idx]['Traduction']
    if src_url is None or src_url is np.nan or tgt_url is None or tgt_url is np.nan:
        print(f'no parallle ressource for {fname}')
        continue

    store_dir = f'{FOLDER_DIR}/mersenne{idx}'
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    # meta
    meta_info = [f"source url: {src_url}\ntarget url: {tgt_url}"]
    
    # EN
    en_fpath = f"{PROCESS_DIR}/EN/{fname}"    
    meta_info_dict = preprocess_to_segment(en_fpath, store_dir, lang = 'en' )
    meta_info = meta_info + [f"tables in EN segments: { ';'.join(meta_info_dict['table'])}"]

    # FR
    fr_fpath = f"{PROCESS_DIR}/FR/{fname}"    
    meta_info_dict = preprocess_to_segment(fr_fpath, store_dir, lang = 'fr' )
    meta_info = meta_info + [f"tables in FR segments: { ';'.join(meta_info_dict['table'])}"]

    # store meta info
    with open(f"{store_dir}/readme.md", 'w', encoding = 'utf-8') as f:
        f.write('\n'.join(meta_info))



### recover to check

recover paragraphs to articles (check)

In [11]:

def recover(f_dir, store_fpath, lang, end_string_list = []):
    # f_dir contains a files for each block as which stored using preprocess_to_segment(fpath, store_dir, lang )
    # get order
    fname_dict = {}
    for fname in os.listdir(f_dir):
        if fname.split('.')[-1] in [lang.lower()]:
            i = int(re.findall(r"to_segment(\d+).+", fname )[0])
            fname_dict[i] = fname
    fname_list = [fname_dict[i] for i in sorted(fname_dict)]
    
    # recover the article
    txt = '\n\n'.join([open(f"{f_dir}/{fname}",'r').read() for fname in fname_list] + end_string_list )
    with open(store_fpath, 'w', encoding = 'utf-8') as f:
        f.write(txt)


def check_recovered_file(origin_fpath, recovered_fpath):
    os.system(f"diff {origin_fpath} {recovered_fpath}")

In [12]:

end_string_list = ['------------------------------------------------------------------------']

for lang in [ 'EN' ,'FR']:
    print(lang)
    for idx in range(len(os.listdir(f"{DATA_DIR}/processed_txt/{lang}"))):
        en_fpath = f'{DATA_DIR}/processed_txt/{lang}/mersenne{idx}.html2txt.processed.txt'
        
        f_dir = f'{DATA_DIR}/to_segment_txt/mersenne{idx}'
        store_fpath = f"{DATA_DIR}/tmp/recover_mersenne{idx}.{lang.lower()}"
        Path(os.path.dirname(store_fpath)).mkdir(parents=True, exist_ok=True)
    
        if not os.path.exists(en_fpath) or not os.path.exists(f_dir):
            print(f'file {en_fpath} not found')
            continue
            
        print(f'\n========{idx}===========')
        recover(f_dir, store_fpath, lang, end_string_list)
        check_recovered_file(en_fpath, store_fpath)

EN

========0===========

========1===========

========2===========

========3===========

========4===========

========5===========

========6===========

========7===========

========8===========

========9===========

========10===========

========11===========

========12===========

========13===========

========14===========

========15===========

========16===========
158,159d157
< 
<                                                                                                                           

========17===========

========18===========

========19===========

========20===========

========21===========

========22===========
FR

========0===========
33,35c33
< Ce numéro spécial est issu du colloque homonyme organisé par l’Académie des sciences à l’Institut de France les 28 et 29 janvier 2020. Ce colloque, coordonné par Sébastien Balibar, Jean Jouzel et Hervé Le Treut, avec le concours de Didier Roux (membres de l’Académie des sciences) a réuni environ 600 p

### extract abstracts

extarct abstracts from html2txt.txt


> We found that the frontend of [Comptes Rendus de l’Académie des Sciences](https://comptes-rendus.academie-sciences.fr/) has changed the way it renders abstracts and titles. We will adapt our scripts later to match the current frontend version in order to extract English–French abstracts and titles.
> The following scripts correspond to the HTML files that we collected on November 5, 2024.


In [13]:
def extract_store_abstract(fpath_dict, store_fpath_dict,  info_dict, to_rm_list = []):
    """fpath : file path for the txt converted from html"""
    # 1. extract the corresponding text spans
    extracted_dict = {}
    for key in ['src', 'tgt']:
        txt = open( fpath_dict[key] ,'r').read()
        txt = re.sub('\xa0', '', txt)
        txt = re.sub('\n\n+', '\n\n', txt)
        txt = unicodedata.normalize("NFC", txt)
    
        begin = False
        for info in info_dict[key]['begin']:
            res = txt.split(info)
            if len(res) >2:
                print(f'to chek\n{info}\n{txt}')
            if len(res) == 2:
                # print('begin')
                txt = res[1].strip()
                begin = True
                break
    
        end = False
        for info in info_dict[key]['end']:
            res = txt.split(info)
            if len(res) == 2:
                # print('end')
                txt = res[0].strip()
                end = True
                break
        if not begin or not end:
            print(f'{key}: no translation of the abstract found in {fpath_dict[key]} ')
            return extracted_dict

        extracted_dict[key] = txt.strip()
        
    # 2. extract the parallel abstracts
    for key in ['src', 'tgt']:
        k2 = 'tgt' if key == 'src' else 'src'
        txt_segments = extracted_dict[key].strip().split('\n\n')
        
        if txt_segments[0].strip() in to_rm_list:
            print(f"extract the abstract for {fpath_dict[key]}")
            
            nb_sents = len(extracted_dict[k2].strip().split('\n\n'))
            if key == 'src':
                extract_txt = '\n\n'.join(txt_segments[1:-nb_sents])
            else:
                extract_txt = '\n\n'.join(txt_segments[1+nb_sents:])
            
            extracted_dict[key] = extract_txt
            # print(f'final {key}:\n{extract_txt}' )
            # usually at most one file contains parallel abstract in our current raw corpus
            break

    # store
    with open(store_fpath_dict['src'] , 'w') as f:
        f.write(extracted_dict['src'])
        
    with open(store_fpath_dict['tgt'] , 'w') as f:
        f.write(extracted_dict['tgt'])

In [14]:
meta_df = pd.read_csv(META_PATH, sep = '\t', index_col = 0 )

TXT_DIR= f"{DATA_DIR}/html2txt"

FOLDER_DIR = f"{DATA_DIR}/to_segment_txt"
idx_pattern = re.compile(r"mersenne(\d+).+")

In [141]:
to_rm_list = ['-   Français\n-   Anglais', '-   Anglais\n-   Français']

src_info_dict = {
    'begin': ['\n\nRésumé\n\n', '\n\nRésumés\n\n' ], 'end': ['\n\nMétadonnées\n\n']
}
tgt_info_dict = {
    'begin': ['\n\nTraduction du résumé\n\n', '\n\nRésumé\n\n' ], 'end': ['\n\nMétadonnées de la traduction\n\n']
}
info_dict = {'src': src_info_dict, 'tgt': tgt_info_dict,  }

    
for fname in os.listdir(f"{TXT_DIR}/EN"):
    if fname in ['.DS_Store']:
        continue
    idx = int(re.findall(idx_pattern, fname)[0])
    store_dir = f'{FOLDER_DIR}/mersenne{idx}'
    print(idx)
    # print(fname)

    src_lang =  meta_df.loc[idx]['LO']
    tgt_lang = 'FR' if src_lang == 'EN' else 'EN'
    fpath_dict = {
        'src': f"{TXT_DIR}/{src_lang}/{fname}",
        'tgt': f"{TXT_DIR}/{tgt_lang}/{fname}",
    }
        
    if not os.path.exists(fpath_dict['src']) or not os.path.exists(fpath_dict['tgt'])   :
            print(f"no translation for {fpath_dict['src']}")
            continue

    store_fpath_dict = {
        'src': f"{store_dir}/abstract.{src_lang.lower()}",
        'tgt': f"{store_dir}/abstract.{tgt_lang.lower()}",
    }

    extract_store_abstract(
        fpath_dict, 
        store_fpath_dict,  
        info_dict, 
        to_rm_list,
    )


9
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne9.html2txt.txt
8
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne8.html2txt.txt
11
no translation for STEP/corpora/raw/Mersenne/html2txt/EN/mersenne11.html2txt.txt
2
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne2.html2txt.txt
5
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne5.html2txt.txt
20
10
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne10.html2txt.txt
17
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne17.html2txt.txt
21
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/EN/mersenne21.html2txt.txt
4
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne4.html2txt.txt
3
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne3.html2txt.txt
15
extract the abstract for STEP/corpora/raw/Mersenne/html2txt/FR/mersenne15.html2txt.txt
12
extract the abstract f

### extarct titles 

In [15]:
import re
src_title_pattern = re.compile(r"%T (.+)")
hyp_title_pattern = re.compile(r".+.  (.+) \(\d\d\d\d\) doi : .+")

In [3]:
meta_df = pd.read_csv(META_PATH, sep = '\t', index_col = 0 )


TXT_DIR= f"{DATA_DIR}/html2txt"

FOLDER_DIR = f"{DATA_DIR}/to_segment_txt"
# ALIGN_DIR = f"{DATA_DIR}/aligned_mersenne_post_processed"

idx_pattern = re.compile(r"mersenne(\d+).+")

begin_info_list = [ 
   "Version originale du texte intégral\n(Proposez une traduction )",
    "Traduction du texte intégral\n(Proposez une traduction dans une autre langue)\n\nTexte intégral traduit par :\n\n",
    "Traduction du texte intégral\n(Proposez une traduction dans une autre langue)",
]


for fname in os.listdir(f"{TXT_DIR}/EN"):
    if fname in ['.DS_Store']:
        continue
    idx = int(re.findall(idx_pattern, fname)[0])
    print(idx)

    src_lang =  meta_df.loc[idx]['LO']
    tgt_lang = 'FR' if src_lang == 'EN' else 'EN'
    fpath_dict = {
        'src': f"{TXT_DIR}/{src_lang}/{fname}",
        'tgt': f"{TXT_DIR}/{tgt_lang}/{fname}",
    }
        
    if not os.path.exists(fpath_dict['src']) or not os.path.exists(fpath_dict['tgt']) :
            print(f"no translation for {fpath_dict['src']}")
            continue

    ##############################################
    # store_dir = f'{ALIGN_DIR}/mersenne{idx}'
    # store_fpath_dict = {
    #     'src': f"{store_dir}/aligned_title.{src_lang.lower()}",
    #     'tgt': f"{store_dir}/aligned_title.{tgt_lang.lower()}",
    # }

    store_dir = f'{FOLDER_DIR}/mersenne{idx}'
    store_fpath_dict = {
        'src': f"{store_dir}/title.{src_lang.lower()}",
        'tgt': f"{store_dir}/title.{tgt_lang.lower()}",
    }

    ##########################################
    txt = open( fpath_dict['tgt'] ,'r').read()
    txt = re.sub('\xa0', '', txt)
    txt = unicodedata.normalize("NFC", txt)

    for info in begin_info_list:
        res = txt.split(info)
        if len(res) >2:
            print(f'to chek\n{info}\n{txt}')
        if len(res) == 2:
            txt = res[0].strip()
            break


    txt = txt.split("\n\n-   BibTeX\n-   RIS\n-   EndNote\n\n")
    txt = txt[1]
    tmp = [l.strip() for l in txt.split('\n') if l.strip()]

    found_src = False
    for l in tmp:
        if not found_src:
            src_title = re.findall(src_title_pattern, l)
            if src_title:
                src_title = src_title[0]
                found_src = True
        else:
            tgt_title = re.findall(hyp_title_pattern, l)
            if tgt_title:
                tgt_title = tgt_title[0]
                break

    # store
    with open(store_fpath_dict['src'] , 'w') as f:
        f.write(src_title)
        
    with open(store_fpath_dict['tgt'] , 'w') as f:
        f.write(tgt_title)

    print(src_lang, tgt_lang)
    print(src_title)
    print(tgt_title)
    


9
FR EN
Changement climatique et biosphère
Climate change and the biosphere
8
FR EN
Énergie et climat: défis et innovations
Energy and climate: challenges and innovations
11
no translation for STEP/corpora/raw/Mersenne/html2txt/EN/mersenne11.html2txt.txt
2
FR EN
Le changement climatique, une chance pour l’humanité?
Climate change, an opportunity for humanity?
5
FR EN
Quelles transformations pour l’atténuation du changement climatique? Des trajectoires d’émissions mondiales à la trajectoire française
Which transformations for climate change mitigation? From global to French emissions pathways
20
EN FR
Glass as a biomaterial: strategies for optimising bioactive glasses for clinical applications
Le verre comme biomatériau : stratégies d'optimisation des verres bioactifs pour les applications cliniques
10
FR EN
Quelles transitions pour l’atténuation du changement climatique? Transformations globales, enjeux sociétaux, et leçons pour la décision
What transitions for climate change mitigatio

## 5. check alignments


>after segmentation using trankit https://trankit.readthedocs.io/en/latest/ 
>
>and alignment using BertAlign https://github.com/bfsujason/bertalign
>
>check the alignments before making the final corpus

In [8]:
import os


ALIGN_DIR = f"{DATA_DIR}/aligned_mersenne"

for dirname in os.listdir(ALIGN_DIR):
    if dirname in ['.DS_Store', 'readme.md', 'README.md']:
        continue
    print(dirname)
    align_dir = f"{ALIGN_DIR}/{dirname}"
    fname_list = ['.'.join(fname.split('.')[:-1]) for fname in os.listdir(align_dir) if fname.split('.')[-1] in ['en', 'fr'] ]
    
    for fname in fname_list:
        en_sents = [ l.strip() for l in open(f"{align_dir}/{fname}.en", 'r').read().split('\n') ]
        fr_sents = [ l.strip() for l in open(f"{align_dir}/{fname}.fr", 'r').read().split('\n') ]

        if len(en_sents) != len(fr_sents):
            print(f'different number of lignes in {dirname}/{fname} with en : {len(en_sents)}, fr : {len(fr_sents)}')
            
        assert(len(en_sents) == len(fr_sents))

        for i in range(len(en_sents)):
            if en_sents[i] in [''] and fr_sents[i] in [''] :
                print(f'empty matches empty in {dirname}/{fname}')
            
            if en_sents[i] in [''] or fr_sents[i] in [''] :
                print(f'align with empty string in {dirname}/{fname}')
        

mersenne3
mersenne4
mersenne5
mersenne2
mersenne22
mersenne13
mersenne14
mersenne15
mersenne12
mersenne24
mersenne23
mersenne7
mersenne0
mersenne9
mersenne8
mersenne1
mersenne6
mersenne19
mersenne21
mersenne17
mersenne10
mersenne20
mersenne18
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string in mersenne18/aligned_segmented52-58
align with empty string i

In [4]:
import os

# TO correct also the following alignments in parallel_corpora for each article
# corrected in parallel_dataset/testset/MERSENNE 
# 838 ⁸ ⁸
# 1729 ² ²
# 3627 - -
# 4232 - -

# for 
# 4366 and et
# we should keep them, as they are actually connecting two equations, which are excluded in our plain text version of MERSENNE articles
# The apparent viscosity can be estimated using the following expressions:
# and
# where 𝜂_(app) is the apparent viscosity, 𝛾 the shear rate, 𝛷 the crystal content (vol%), and AS the crystal’s aspect ratio.


DATA_DIR = 'STEP/corpora/raw/Mersenne/sharedoc-testsets-MERSENNE'


en_fpath = f"{DATA_DIR}/MERSENNE_sent.en"
fr_fpath = f"{DATA_DIR}/MERSENNE_sent.fr"

en_sents = [ l.strip() for l in open(en_fpath, 'r').read().split('\n') ]
fr_sents = [ l.strip() for l in open(fr_fpath, 'r').read().split('\n') ]
assert(len(en_sents) ==len(fr_sents))


for i in range(len(en_sents)):
    if len(en_sents[i]) < 10 or len(fr_sents[i]) < 10 :
            # print(f'too short sentence-level segments')
            print(i, en_sents[i], fr_sents[i])

65 France France
80 France France
82 Table of contents Sommaire
494 3.3. Acting 3.3. Agir
763 Funding Financement
838 ⁸ ⁸
1093 They are: Il s’agit de:
1729 ² ²
2682 — Gallium — le gallium
2710 — Xenon — le xénon
2953 Actinides Les actinides
3407 Prologue Prologue
3627 - -
4050 [2009]. [2009].
4054 [2008]. [2008].
4061 [2019]. [2019].
4216 Dedication Dédicace
4232 - -
4366 and et


## 6. store test sets in txt

put isolated equations and tables aside

In [2]:

import pandas as pd
# key = article id, values = segment id for equations and tables
equation_dict = { 15 : [25], 18: [11,14], 22: [10,13, 17,61,63, 71] }
table_dict = {
    6: ["4-6;7-9;10-12;47-49;53-55;58-60"],
    12: ["31-33"],
    16: ["50-58"], # 18 in fr ["50-52"] in to_segment_txt
    17: ["33-35;46-48;84-86"],
    20: ["4-6;20-22;23-25;40-42"],
    22: ["36-58"]
 }

def get_table_idx(meta_info):
    # e.g. meta_info = "4-6;7-9;10-12;47-49;53-55;58-60", where 4 contains table1. 5 contains caption and 6 contains the table content
    if meta_info == '':
        return []
    res = [ l.split('-') for l in meta_info.split(';')]
    res = [ i  for s in res for i in range( int(s[0]), int(s[1])+1)  ]
    return res

In [4]:



RES_DIR = f"{DATA_DIR}/eval_mersenne"
FOLDER_DIR = f"{DATA_DIR}/to_segment_txt" # for titles
STORE_DIR = f"{DATA_DIR}/txt"

sents_store_dir = f"{STORE_DIR}/sents"
Path(sents_store_dir).mkdir(parents=True, exist_ok=True)

doc_store_dir = f"{STORE_DIR}/blocks_doc"
Path(doc_store_dir).mkdir(parents=True, exist_ok=True)

doc_sep_store_dir = f"{STORE_DIR}/blocks_sep"
Path(doc_sep_store_dir).mkdir(parents=True, exist_ok=True)


for idx in range(23):
    eval_fpath = f"{RES_DIR}/Resumes_MERSENNE{idx}_eval_model_monotransquest-da-any_en.tsv"
    res_df = pd.read_csv(eval_fpath, sep = '\t', index_col = 0)
    fname_list = [ l for l in set(res_df['doc_id']) if l != 'aligned_abstract']

    # reorder segments
    fname_dict = {}
    if idx > 0:
        fname_dict[-1] = 'aligned_abstract'
    for fname in fname_list:
        i = int(re.findall(r"aligned_segmented(\d+)", fname )[0])

        if i in equation_dict.get(idx, []) or i in get_table_idx(table_dict.get(idx, [''])[0]):
            print('equation and tables ')
            continue
            
        fname_dict[i] = fname
        
    fname_list = [fname_dict[i] for i in sorted(fname_dict)]

    title_fname = f"{FOLDER_DIR}/mersenne{idx}/title"
    if not os.path.exists(f"{title_fname}.en") or not os.path.exists(f"{title_fname}.fr"):
        print(f"no parallel title for mersenne {idx}")
        en_title = ''
        fr_title = ''
    else:
        en_title = open(f"{title_fname}.en").read().strip()
        fr_title = open(f"{title_fname}.fr").read().strip()
    
    # collect segments
    en_doc_list = [en_title]
    fr_doc_list = [fr_title]

    en_doc_sep_list = [en_title]
    fr_doc_sep_list = [fr_title]
    
    en_sents_list = [en_title]
    fr_sents_list = [fr_title]
    for fname in fname_list:
    
        doc_df = res_df[res_df['doc_id'] ==  fname]
    
        en_sents_list += ['\n'.join(doc_df['tgt'].values)]
        fr_sents_list += ['\n'.join(doc_df['src'].values)]
    
        en_doc_list += [' '.join(doc_df['tgt'].values)]
        fr_doc_list += [' '.join(doc_df['src'].values)]

        en_doc_sep_list += ['<sep> '.join(doc_df['tgt'].values)]
        fr_doc_sep_list += ['<sep> '.join(doc_df['src'].values)]

   
    # store sents
    score_fname = f"{sents_store_dir}/MERSENNE{idx}"
    with open( f"{score_fname}.en.txt", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_sents_list).strip() )

    with open( f"{score_fname}.fr.txt", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_sents_list).strip() )

    # store doc
    score_fname = f"{doc_store_dir}/MERSENNE{idx}"
    with open( f"{score_fname}.en.txt", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_doc_list).strip() )

    with open( f"{score_fname}.fr.txt", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_doc_list).strip() )

    # store doc_sep
    score_fname = f"{doc_sep_store_dir}/MERSENNE{idx}"
    with open( f"{score_fname}.en.txt", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_doc_sep_list).strip() )

    with open( f"{score_fname}.fr.txt", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_doc_sep_list).strip() )
        


### tables & equations

In [ ]:

import os

FOLDER_DIR = f"{DATA_DIR}/to_segment_txt"
STORE_DIR = f"{DATA_DIR}/txt"

eq_store_dir = f"{STORE_DIR}/equations"
Path(eq_store_dir).mkdir(parents=True, exist_ok=True)

tab_store_dir = f"{STORE_DIR}/tables"
Path(tab_store_dir).mkdir(parents=True, exist_ok=True)

for k, v in equation_dict.items():
    store_dir = f"{eq_store_dir}/mersenne{k}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    
    for eq in v:
        os.system(f"cp {FOLDER_DIR}/mersenne{k}/to_segment{eq}.en {store_dir}/to_segment{eq}.en")


for idx in range(23):
    if idx not in table_dict:
        continue
    doc_dir = f"{FOLDER_DIR}/mersenne{idx}"
    fname_list = [ l for l in os.listdir(doc_dir) if l not in [ 'readme.md', 'abstract.en', 'abstract.fr', '.DS_Store' ]]
    
    # filter tables
    table_list = []
    for fname in fname_list:
        # print(fname)
        i = int(re.findall(r"to_segment(\d+).+", fname )[0])
        
        if  i in get_table_idx(table_dict.get(idx, [''])[0]):
            table_list.append(fname)

    store_dir = f"{tab_store_dir}/mersenne{idx}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)

    for fname in table_list:
        os.system(f"cp {doc_dir}/{fname} {store_dir}/{fname}")
        

### meta info

In [5]:

FOLDER_DIR = f"{DATA_DIR}/to_segment_txt" # for equations and tables


res_ls = []
for idx in range(23):

    fpath = f"{FOLDER_DIR}/mersenne{idx}/readme.md"
    res_ls.append( '\n'.join([f"mersenne{idx}"] + open(fpath, 'r').read().strip().split('\n')[:2]) )
    
print('\n\n'.join(res_ls))    
    

mersenne0
source url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/fr/10.5802/crgeos.31/
target url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/en/10.5802/crgeos.31/

mersenne1
source url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/10.5802/crgeos.29/
target url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/en/10.5802/crgeos.29/

mersenne2
source url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/10.5802/crgeos.28/
target url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/en/10.5802/crgeos.28/

mersenne3
source url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/10.5802/crgeos.26/
target url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/en/10.5802/crgeos.26/

mersenne4
source url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/10.5802/crgeos.24/
target url: https://comptes-rendus.academie-sciences.fr/geoscience/articles/

### parallel_corpora per blocks

In [11]:
FOLDER_DIR = f"{DATA_DIR}/to_segment_txt" # for equations and tables

RES_DIR = f"{DATA_DIR}/eval_mersenne"
STORE_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-block"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)

# txt
for idx in range(23):
    eval_fpath = f"{RES_DIR}/Resumes_MERSENNE{idx}_eval_model_monotransquest-da-any_en.tsv"
    res_df = pd.read_csv(eval_fpath, sep = '\t', index_col = 0)
    fname_list = [ l for l in set(res_df['doc_id']) if l != 'aligned_abstract']

    # reorder segments
    fname_dict = {}
    if idx > 0:
        fname_dict[-1] = 'aligned_abstract'
    for fname in fname_list:
        i = int(re.findall(r"aligned_segmented(\d+)", fname )[0])

        if i in equation_dict.get(idx, []) or i in get_table_idx(table_dict.get(idx, [''])[0]):
            print('equation and tables ')
            continue
            
        fname_dict[i] = fname
        
    fname_list = [fname_dict[i] for i in sorted(fname_dict)]
    
    # store segments
    for fname in fname_list:
        doc_df = res_df[res_df['doc_id'] ==  fname]    
        store_dir = f"{STORE_DIR}/mersenne{idx}/{fname}"
        Path(store_dir).mkdir(parents=True, exist_ok=True)
        
        # store sents
        score_fname = f"{store_dir}/sent"
        with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
            f.write( '\n'.join(doc_df['tgt'].values) )

        with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
            f.write( '\n'.join(doc_df['src'].values))

        # store doc
        score_fname = f"{store_dir}/block"
        with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
            f.write( ' '.join(doc_df['tgt'].values))

        with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
            f.write(' '.join(doc_df['src'].values) )




### parallel_corpora per article

In [6]:


RES_DIR = f"{DATA_DIR}/eval_mersenne"
sep_tag = '<sep>'

STORE_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-article"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)


for idx in range(23):
    eval_fpath = f"{RES_DIR}/Resumes_MERSENNE{idx}_eval_model_monotransquest-da-any_en.tsv"
    res_df = pd.read_csv(eval_fpath, sep = '\t', index_col = 0)
    fname_list = [ l for l in set(res_df['doc_id']) if l != 'aligned_abstract']

    # reorder segments
    fname_dict = {}
    if idx > 0:
        fname_dict[-1] = 'aligned_abstract'
    for fname in fname_list:
        i = int(re.findall(r"aligned_segmented(\d+)", fname )[0])

        if i in equation_dict.get(idx, []) or i in get_table_idx(table_dict.get(idx, [''])[0]):
            print('equation and tables ')
            continue
            
        fname_dict[i] = fname
        
    fname_list = [fname_dict[i] for i in sorted(fname_dict)]

    store_dir = f"{STORE_DIR}/mersenne{idx}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    # print(fname_list)
    
    title_fname = f"{FOLDER_DIR}/mersenne{idx}/title"
    if not os.path.exists(f"{title_fname}.en") or not os.path.exists(f"{title_fname}.fr"):
        print(f"no parallel title for mersenne {idx}")
        en_title = ''
        fr_title = ''
    else:
        en_title = open(f"{title_fname}.en").read().strip()
        fr_title = open(f"{title_fname}.fr").read().strip()
    
    # collect segments
    en_doc_list = [en_title]
    fr_doc_list = [fr_title]

    en_doc_sep_list = [en_title]
    fr_doc_sep_list = [fr_title]
    
    en_sents_list = [en_title]
    fr_sents_list = [fr_title]
    for fname in fname_list:
    
        doc_df = res_df[res_df['doc_id'] ==  fname]
    
        en_sents_list += ['\n'.join(doc_df['tgt'].values)]
        fr_sents_list += ['\n'.join(doc_df['src'].values)]
    
        en_doc_list += [' '.join(doc_df['tgt'].values)]
        fr_doc_list += [' '.join(doc_df['src'].values)]

        en_doc_sep_list += [f'{sep_tag} '.join(doc_df['tgt'].values)]
        fr_doc_sep_list += [f'{sep_tag} '.join(doc_df['src'].values)]
        
    # store sents
    score_fname = f"{store_dir}/sent"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_sents_list) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_sents_list))

    # store doc
    score_fname = f"{store_dir}/block"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_doc_list) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_doc_list) )

    # store doc_sep
    sep_store_dir = f"{store_dir}/doc_auxiliaire"
    Path(sep_store_dir).mkdir(parents=True, exist_ok=True)
    score_fname = f"{sep_store_dir}/block_sep"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_doc_sep_list) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_doc_sep_list) )
        


### collect together

In [3]:

TXT_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-article"

STORE_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-testset"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)
sep_tag = '<sep>'


en_sents_ls = []
fr_sents_ls = []

en_doc_ls = []
fr_doc_ls = []
en_doc_sep_ls = []
fr_doc_sep_ls = []

en_article_ls = []
fr_article_ls = []
en_article_sep_ls = []
fr_article_sep_ls = []

for idx in range(23):
    data_dir = f"{TXT_DIR}/mersenne{idx}"
    doc_en = open(f"{data_dir}/block.en").read().strip()
    doc_fr = open(f"{data_dir}/block.fr").read().strip()

    sent_en = open(f"{data_dir}/sent.en").read().strip()
    sent_fr = open(f"{data_dir}/sent.fr").read().strip()
    
    en_doc_ls.append( doc_en)
    fr_doc_ls.append( doc_fr)
    en_sents_ls.append( sent_en)
    fr_sents_ls.append( sent_fr)
    
    en_article_ls.append( ' '.join( doc_en.split('\n'))  )
    fr_article_ls.append( ' '.join( doc_fr.split('\n'))  )

    en_article_sep_ls.append( f'{sep_tag} '.join( sent_en.split('\n'))  )
    fr_article_sep_ls.append( f'{sep_tag} '.join( sent_fr.split('\n'))  )

    en_doc_sep_ls.append( open(f"{data_dir}/doc_auxiliaire/block_sep.en").read().strip())
    fr_doc_sep_ls.append( open(f"{data_dir}/doc_auxiliaire/block_sep.fr").read().strip())
    
          
    # store sents
    score_fname = f"{STORE_DIR}/MERSENNE_sent"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_sents_ls) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_sents_ls))

    # store doc
    score_fname = f"{STORE_DIR}/MERSENNE_block"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_doc_ls) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_doc_ls) )

    # store doc_sep
    sep_store_dir = f"{STORE_DIR}/doc_auxiliaire"
    Path(sep_store_dir).mkdir(parents=True, exist_ok=True)
    
    score_fname = f"{sep_store_dir}/MERSENNE_block_sep"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_doc_sep_ls) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_doc_sep_ls) )

    # store doc
    score_fname = f"{STORE_DIR}/MERSENNE_article"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_article_ls) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_article_ls) )


    score_fname = f"{sep_store_dir}/MERSENNE_article_sep"
    with open( f"{score_fname}.en", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(en_article_sep_ls) )

    with open( f"{score_fname}.fr", 'w', encoding = 'utf-8' ) as f:
        f.write( '\n'.join(fr_article_sep_ls) )
    

    
        

## 7. xml/txt by collection

In [6]:
import xml.etree.ElementTree as ET
import unicodedata
from pathlib import Path
import os
import pandas as pd
import re
ET.VERSION


def text2tmx_mersenne(meta_df, idx, aligned_fname, store_path, title_en, title_fr, name = 'MERSENNE', date = '2024-11-15'):
    # src = 'FR', tgt = 'EN', 
    # aligned sentences are in {aligned_fname}.{src} and {aligned_fname}.{tgt}
    src = meta_df.loc[idx]['LO'].upper()
    tgt = 'EN' if src == 'FR' else 'FR'

    docid = meta_df.loc[idx]['docid']
    # the src and target sentences
    with open(f"{aligned_fname}.{src.lower()}", 'rt', encoding='utf-8') as f:
        doc_sep_src = [ l for l in f.read().strip().split('\n') if l.strip()]
    
    with open(f"{aligned_fname}.{tgt.lower()}", 'rt', encoding='utf-8') as f:
        doc_sep_tgt = [ l for l in f.read().strip().split('\n') if l.strip()]
        
    assert(len(doc_sep_src) == len(doc_sep_tgt))

    # Create the root element of the TMX document
    tmx = ET.Element('tmx', version='1.4b')

    header_attrib = { 
        'creationtool' : 'xml.etree.ElementTree',
        'creationtoolversion' : '1.3.0',
        'datatype': "PlainText",
        'segtype' : "sentence",
        'adminlang': "en",
        'srclang': src, 
        'tgtlang': tgt,
        'o-tmf': "XML",
        'creationdate': date,
        'creationid': "MaTOS"
    }

    header = ET.SubElement(tmx, 'header', header_attrib)

    # Set the language codes for the source and target languages
    etm = ET.SubElement(header, 'note')
    etm.text = f"This is the sentence alignement file for {name}-{docid}." + " segId begin by 1, tuid = segId" 
    etm = ET.SubElement(header, 'docid')
    etm.text = docid
    # url_src
    etm = ET.SubElement(header, 'url_src')
    etm.text = meta_df.loc[idx]['Version originale']
    # url_src
    etm = ET.SubElement(header, 'url_tgt')
    etm.text = meta_df.loc[idx]['Traduction']
    # author
    etm = ET.SubElement(header, 'author')
    etm.text = unicodedata.normalize('NFC', re.sub('\xa0', '', meta_df.loc[idx]['Auteur']))
    # title_en
    etm = ET.SubElement(header, 'title_en')
    etm.text = title_en  
    # title_fr
    etm = ET.SubElement(header, 'title_fr')
    etm.text = title_fr
    # volume
    etm = ET.SubElement(header, 'volume')
    # meta_df.loc[0][['Série', 'Année', 'Vol.', 'Thém.']].values
    etm.text =  f"{meta_df.loc[idx]['Série']}, {meta_df.loc[idx]['Année']}, Volume {meta_df.loc[idx]['Vol.']}, {meta_df.loc[idx]['Thém.']}"
    # translator
    etm = ET.SubElement(header, 'translator')
    etm.text = unicodedata.normalize('NFC', re.sub('\xa0', '', meta_df.loc[idx]['Traducteur.trice']))
    # language code
    etm = ET.SubElement(header, 'elem', type='sourceLanguage')
    etm.text = src
    etm = ET.SubElement(header, 'elem', type='targetLanguage')
    etm.text = tgt

    # Create a new body element for the translation units
    body = ET.SubElement(tmx, 'body')
    
    seg_id = 1
    XML_LANG = '{http://www.w3.org/XML/1998/namespace}lang'
    # Iterate through the sentence pairs and add them to the TMX document

    for p_id, src_block in enumerate(doc_sep_src, start=1):
        sents_src = [l.strip() for l in src_block.strip().split('<sep>') if l.strip()]
        sents_tgt = [l.strip() for l in doc_sep_tgt[p_id-1].strip().split('<sep>') if l.strip()]
        assert len(sents_src) == len(sents_tgt)
        for s, t in zip(sents_src, sents_tgt):    
            # id with paragraph boundaries
            tu = ET.SubElement(body, 'tu', {'tuid':str(seg_id)})
            prop = ET.SubElement(tu, "prop", type="paragraph")
            prop.text = str(p_id)

            # sentences
            tuv_src = ET.SubElement(tu, 'tuv', {XML_LANG : src})
            seg_src = ET.SubElement(tuv_src, 'seg')
            seg_src.text = unicodedata.normalize('NFC', s )
    
            tuv_tgt = ET.SubElement(tu, 'tuv', {XML_LANG :tgt})
            seg_tgt = ET.SubElement(tuv_tgt, 'seg')
            seg_tgt.text = unicodedata.normalize('NFC', t )
            
            seg_id +=1


    # Write the TMX document to a file
    tree = ET.ElementTree(tmx)
    ET.indent(tree, space="    ", level=0)
    with open(store_path, 'wb') as f : 
        tree.write(f, encoding='utf-8', xml_declaration=True)
        



In [ ]:
date = '2024-11-15'


meta_df = pd.read_csv(META_PATH, sep = '\t', index_col = 0 )

TXT_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-article"
ALIGN_DIR = f"{DATA_DIR}/aligned_mersenne_post_processed"

STORE_DIR = f"{DATA_DIR}/tmx/Mersenne-groupby-article-final"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)


print(len(meta_df))
for idx in range(len(meta_df)):
    docid = meta_df.loc[idx]['docid']
    
    tmp_n = 0
    if idx > 10:
        tmp_n = 1
    if idx > 14:
        tmp_n = 2
    print(idx, idx+tmp_n)
    
    
    title_en = open(f"{ALIGN_DIR}/mersenne{idx+tmp_n}/aligned_title.en", 'r').read().strip()
    title_fr = open(f"{ALIGN_DIR}/mersenne{idx+tmp_n}/aligned_title.fr", 'r').read().strip()

    store_dir = f"{STORE_DIR}/{docid}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)


    aligned_fname = f"{TXT_DIR}/{docid}/doc_auxiliaire/block_sep"
    store_path = f"{store_dir}/MERSENNE_aligned.tmx"
        
    text2tmx_mersenne(
        meta_df, 
        idx, 
        aligned_fname, 
        store_path, 
        title_en, 
        title_fr, 
        name = 'MERSENNE', 
        date = date
    )
        
    try:
        tree = ET.parse(store_path, ET.XMLParser(encoding='utf-8'))
    except:
        print(f"Cannot parse, potentially no text in {fpath}")
        warnings.warn(f"Cannot parse file {fpath} as tree")


### sent

In [10]:
date = '2024-11-15'

meta_df = pd.read_csv(META_PATH, sep = '\t', index_col = 0 )

ALIGNED_PAPER_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-article"
STORE_DIR = f"{DATA_DIR}/tmx/Mersenne-groupby-article"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)

ALIGN_DIR = f"{DATA_DIR}/aligned_mersenne_post_processed"


for idx in range(23):
    
    store_dir = f"{STORE_DIR}/mersenne{idx}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    
    aligned_fname = f"{ALIGNED_PAPER_DIR}/mersenne{idx}/sent"
    author1 = re.sub( '\xa0', '', meta_df.loc[idx]['Auteur'].split(',',1)[0])
    author1 = re.sub(' ', '-', author1)
    store_path = f"{store_dir}/{author1}-{meta_df.loc[idx]['Année']}_sent.tmx"

    title_en = open(f"{ALIGN_DIR}/mersenne{idx}/aligned_title.en", 'r').read().strip()
    title_fr = open(f"{ALIGN_DIR}/mersenne{idx}/aligned_title.fr", 'r').read().strip()
    text2tmx_mersenne(
        meta_df, 
        idx, 
        aligned_fname, 
        store_path, 
        title_en, 
        title_fr, 
        name = 'MERSENNE', 
        date = date,
    )

    try:
        tree = ET.parse(store_path, ET.XMLParser(encoding='utf-8'))
    except:
        print(f"Cannot parse, potentially no text in {fpath}")
        warnings.warn(f"Cannot parse file {fpath} as tree")


### block_sep

In [11]:

# block sep
date = '2024-11-15'

meta_df = pd.read_csv(META_PATH, sep = '\t', index_col = 0 )

ALIGNED_PAPER_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-article"
STORE_DIR = f"{DATA_DIR}/tmx/Mersenne-groupby-article"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)

ALIGN_DIR = f"{DATA_DIR}/aligned_mersenne_post_processed"


for idx in range(23):
    store_dir = f"{STORE_DIR}/mersenne{idx}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    
    aligned_fname = f"{ALIGNED_PAPER_DIR}/mersenne{idx}/doc_auxiliaire/block_sep"
    author1 = re.sub( '\xa0', '', meta_df.loc[idx]['Auteur'].split(',',1)[0])
    author1 = re.sub(' ', '-', author1)
    store_path = f"{store_dir}/{author1}-{meta_df.loc[idx]['Année']}_block_sep.tmx"

    title_en = open(f"{ALIGN_DIR}/mersenne{idx}/aligned_title.en", 'r').read().strip()
    title_fr = open(f"{ALIGN_DIR}/mersenne{idx}/aligned_title.fr", 'r').read().strip()
    text2tmx_mersenne(
        meta_df, 
        idx, 
        aligned_fname, 
        store_path, 
        title_en, 
        title_fr, 
        name = 'MERSENNE', 
        date = date,
    )

    try:
        tree = ET.parse(store_path, ET.XMLParser(encoding='utf-8'))
    except:
        print(f"Cannot parse, potentially no text in {fpath}")
        warnings.warn(f"Cannot parse file {fpath} as tree")


In [12]:

# block
date = '2024-11-15'

meta_df = pd.read_csv(meta_path, sep = ';' )

ALIGNED_PAPER_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-article"
STORE_DIR = f"{DATA_DIR}/tmx/Mersenne-groupby-article"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)

ALIGN_DIR = f"{DATA_DIR}/aligned_mersenne_post_processed"


for idx in range(23):


    store_dir = f"{STORE_DIR}/mersenne{idx}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)
    
    aligned_fname = f"{ALIGNED_PAPER_DIR}/mersenne{idx}/block"
    author1 = re.sub( '\xa0', '', meta_df.loc[idx]['Auteur'].split(',',1)[0])
    author1 = re.sub(' ', '-', author1)
    store_path = f"{store_dir}/{author1}-{meta_df.loc[idx]['Année']}_block.tmx"

    title_en = open(f"{ALIGN_DIR}/mersenne{idx}/aligned_title.en", 'r').read().strip()
    title_fr = open(f"{ALIGN_DIR}/mersenne{idx}/aligned_title.fr", 'r').read().strip()
    text2tmx_mersenne(
        meta_df, 
        idx, 
        aligned_fname, 
        store_path, 
        title_en, 
        title_fr, 
        name = 'MERSENNE', 
        date = date,
    )

    try:
        tree = ET.parse(store_path, ET.XMLParser(encoding='utf-8'))
    except:
        print(f"Cannot parse, potentially no text in {fpath}")
        warnings.warn(f"Cannot parse file {fpath} as tree")
